# **ILP Meal Planner**

This notebook tests the `ILPMealPlanner`, which uses Integer Linear Programming (via [PuLP](https://coin-or.github.io/pulp/) and the CBC solver) to find a provably near-optimal 7-day meal plan.

The ILP maximises the same composite objective as the GA:
$$\text{fitness} = \text{pantry\_score} - \text{dietary\_penalty} - \text{waste\_penalty} - \text{budget\_penalty}$$

It serves as an **oracle / upper bound** when evaluating other planners — any other planner's fitness score can be compared against the ILP to compute an *optimality gap*.

In [1]:
import json
import os
import sys

sys.path.insert(0, os.path.abspath(".."))

from datetime import datetime, timedelta
from random import randint, random, sample, seed

from pulp import LpStatus

from engines import ILPMealPlanner, get_pantry_ingredient, load_all_ingredients, make_preferences
from models import DietaryTag, MealPlanningEnvironment, NutritionalInformation, Pantry, Recipe

In [2]:
seed(1)

In [3]:
preferences = make_preferences()

In [4]:
all_ingredients = load_all_ingredients("../recipe_extraction/supplemented_structured_ingredients.json")

In [5]:
CURRENT_DATE = datetime.now()

In [6]:
pantry_ingredient_name_by_quantity = {
    "chicken breast": 800,
    "broccoli": 1500,
    "rice": 1000,
}

In [7]:
pantry_ingredients = [
    get_pantry_ingredient(name, CURRENT_DATE + timedelta(days=randint(1, 7)), all_ingredients)
    for name in pantry_ingredient_name_by_quantity.keys()
]

In [8]:
pantry_ingredient_by_quantity = dict(zip(pantry_ingredients, pantry_ingredient_name_by_quantity.values()))

In [9]:
pantry = Pantry()

for pantry_ingredient, quantity in pantry_ingredient_by_quantity.items():
    pantry.add(
        pantry_ingredient,
        quantity,
    )

In [10]:
pantry.print()

---
Quantity: 800 g
Ingredient: chicken breast
	Estimated Expiration Date: 2026-05-01
	Nutritional Information:
		Calories: 125.0 kcal
		Carbohydrates: 1.79 g
		Sugar: 0.0 g
		Protein: 16.07 g
		Fat: 5.36 g
		Saturated Fat: 1.79 g
		Fiber: 1.8 g
		Sodium: 571.0 mg
		Gluten Free: Yes
		Lactose Free: Yes
		Vegetarian: No
		Vegan: No
---
---
Quantity: 1500 g
Ingredient: broccoli
	Estimated Expiration Date: 2026-05-04
	Nutritional Information:
		Calories: 31.0 kcal
		Carbohydrates: 6.27 g
		Sugar: 1.4 g
		Protein: 2.57 g
		Fat: 0.34 g
		Saturated Fat: 0.039 g
		Fiber: 2.4 g
		Sodium: 36.0 mg
		Gluten Free: Yes
		Lactose Free: Yes
		Vegetarian: Yes
		Vegan: Yes
---
---
Quantity: 1000 g
Ingredient: rice
	Estimated Expiration Date: 2026-05-06
	Nutritional Information:
		Calories: 356.0 kcal
		Carbohydrates: 80.0 g
		Sugar: 0.0 g
		Protein: 6.67 g
		Fat: 0.0 g
		Saturated Fat: 0.0 g
		Fiber: 2.2 g
		Sodium: 0.0 mg
		Gluten Free: Yes
		Lactose Free: Yes
		Vegetarian: Yes
		Vegan: Yes
---


In [11]:
with open("../recipe_extraction/supplemented_structured_recipes.json", "r") as file:
    all_recipes = json.load(file)

In [12]:
NUM_EXTRA_RECIPES = 10

In [13]:
filtered_recipes_indices = []

for ingredient_name in pantry_ingredient_name_by_quantity.keys():
    for i in range(len(all_recipes)):
        if ingredient_name in [ingredient["ingredient"] for ingredient in all_recipes[i]["ingredients"]]:
            filtered_recipes_indices.append(i)

# sample some more recipes that do not contain any ingredients from the pantry
sampled_recipes_indices = sample(
    [i for i in range(len(all_recipes)) if i not in filtered_recipes_indices],
    NUM_EXTRA_RECIPES,
)

In [14]:
assert len(set(filtered_recipes_indices).intersection(set(sampled_recipes_indices))) == 0

final_recipes = [all_recipes[i] for i in filtered_recipes_indices + sampled_recipes_indices]

In [15]:
tag_map = {
    "VEGETARIAN": DietaryTag.VEGETARIAN,
    "VEGAN": DietaryTag.VEGAN,
    "GLUTEN_FREE": DietaryTag.GLUTEN_FREE,
    "LACTOSE_FREE": DietaryTag.LACTOSE_FREE,
}

recipe_objects = []

for raw_recipe in final_recipes:
    ingredients = {ingredient["ingredient"]: ingredient["quantity"] for ingredient in raw_recipe["ingredients"]}
    dietary_tags = [tag_map[tag] for tag in raw_recipe.get("dietary_tags", []) if tag in tag_map]
    nutritional_information = raw_recipe["nutritional_information"]

    recipe = Recipe(
        name=raw_recipe["name"].strip(),
        ingredients=ingredients,
        dietary_tags=dietary_tags,
        instructions=raw_recipe.get("instructions", []),
    )

    recipe.nutritional_information = NutritionalInformation(
        calories=nutritional_information.get("calories"),
        carbohydrates=nutritional_information.get("carbohydrates"),
        sugar=nutritional_information.get("sugar"),
        protein=nutritional_information.get("protein"),
        fat=nutritional_information.get("fat"),
        saturated_fat=nutritional_information.get("saturated_fat"),
        fiber=nutritional_information.get("fiber"),
        sodium=nutritional_information.get("sodium"),
        is_gluten_free=nutritional_information.get("is_gluten_free"),
        is_lactose_free=nutritional_information.get("is_lactose_free"),
        is_vegetarian=nutritional_information.get("is_vegetarian"),
        is_vegan=nutritional_information.get("is_vegan"),
    )

    recipe_objects.append(recipe)

final_recipes = recipe_objects

In [16]:
print(f"Number of recipes before filtering: {len(all_recipes)}")
print(f"Number of recipes after filtering: {len(final_recipes)}")

Number of recipes before filtering: 19716
Number of recipes after filtering: 138


In [17]:
meal_planning_environment = MealPlanningEnvironment(
    recipes=final_recipes,
    pantry=pantry,
    preferences=preferences,
)

In [18]:
all_ingredient_names = []

for recipe in meal_planning_environment.recipes:
    for ingredient_name in recipe.ingredients.keys():
        all_ingredient_names.append(ingredient_name)

In [19]:
all_ingredient_costs = {}

for ingredient_name in sorted(set(all_ingredient_names)):
    all_ingredient_costs[ingredient_name] = random()

In [20]:
meal_planning_environment.ingredient_costs = all_ingredient_costs
meal_planning_environment._check_ingredient_costs()

In [21]:
planner = ILPMealPlanner(meal_planning_environment)

In [22]:
best_meal_plan, fitness_score = planner.generate_meal_plan(time_limit=120, mip_gap=0.1, msg=True)

In [23]:
print(f"Solve status : {LpStatus[planner.solve_status]}")
print(f"Fitness score: {fitness_score:.4f}")

Solve status : Optimal
Fitness score: -0.0260


In [24]:
pantry_consumption_df = planner.get_pantry_consumption()
pantry_consumption_df

,Ingredient,Available,Consumed,Unused,Expires in
0,chicken breast,800,226.796185,573.203815,1d
1,broccoli,1500,1500.000000,0.000000,4d
2,rice,1000,863.828571,136.171429,6d


In [25]:
meal_plan_df = planner.get_meal_plan_recipes()
meal_plan_df

,Day,Meal 1,Meal 2,Meal 3
0,1,Orange Chicken,Orange Chicken,Roast Turkey with Pesto-Rice Stuffing
1,2,Cream of Broccoli Soup,Roasted Broccoli Florets with Toasted Breadcru...,Moroccan Chicken with Kumquats and Prunes
2,3,Cream of Broccoli Soup,"Fried Rice with Ham, Egg, and Scallions","Fried Rice with Ham, Egg, and Scallions"
3,4,Cream of Broccoli Soup with Wild Mushrooms,Broccoli with Garlic and Parmesan Cheese,Roast Turkey with Pesto-Rice Stuffing
4,5,Harissa Shrimp And Summer Vegetable Sauté,Yukari Shiso Salt Yaki Onigiri,Moroccan Chicken with Kumquats and Prunes
5,6,Harissa Shrimp And Summer Vegetable Sauté,Yukari Shiso Salt Yaki Onigiri,Moroccan Chicken with Kumquats and Prunes
6,7,Green Poblano Rice (Arroz Verde al Poblano),Roast Turkey with Pesto-Rice Stuffing,Mussels with Parsley and Garlic


In [26]:
shopping_list_df, num_ingredients, total_cost = planner.get_shopping_list()

print(f"Total ingredients needed to purchase: {num_ingredients}")
print(f"Total cost: \u20ac{total_cost:.2f}")

shopping_list_df

Total ingredients needed to purchase: 68
Total cost: €51.37


,Ingredient,Quantity to Buy (g),Cost (€)
0,Bacon slices,11.1,0.00
1,Olive oil,11.1,0.02
2,Parmesan cheese,12.5,0.05
3,Salt,2.9,0.01
4,black pepper,0.3,0.00
...,...,...,...
64,white onion,14.3,0.03
65,wild mushrooms,94.5,0.58
66,yakari shiso salt,2.8,0.00
67,zucchini,7.1,0.02


In [27]:
daily_nutrition_df = planner.get_daily_nutrition()
daily_nutrition_df

,Calories,Protein,Δ Calories and Target Calories,Δ Protein and Target Protein
Day 1,1962.2 kcal,49.6 g,-37.8 kcal,-0.4 g
Day 2,1979.5 kcal,51.1 g,-20.5 kcal,1.1 g
Day 3,1137.9 kcal,42.4 g,-862.1 kcal,-7.6 g
Day 4,2238.1 kcal,51.6 g,238.1 kcal,1.6 g
Day 5,2028.3 kcal,49.6 g,28.3 kcal,-0.4 g
Day 6,2028.3 kcal,49.6 g,28.3 kcal,-0.4 g
Day 7,1979.2 kcal,50.8 g,-20.8 kcal,0.8 g
